# AnnNet Showcase

This notebook is a compact tour of the current public API, with particular emphasis on the edge model.

The central fact to keep in mind is this: `add_edge(...)` is not just for binary edges. It is the unified structural constructor for the graph.


In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / 'annnet').exists():
    for parent in repo_root.parents:
        if (parent / 'annnet').exists():
            repo_root = parent
            break
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import annnet as an


In [2]:
G = an.AnnNet(directed=True)
G.add_vertices_bulk([
    ('A', {'kind': 'gene'}),
    ('B', {'kind': 'gene'}),
    ('C', {'kind': 'gene'}),
    ('D', {'kind': 'metabolite'}),
    ('E', {'kind': 'complex'}),
])


## Binary, hyper, stoichiometric, and edge-entity cases

All of these go through `add_edge(...)`, but the endpoint shapes differ.


In [3]:
# binary
G.add_edge('A', 'B', edge_id='e_bin', directed=True, relation='activates')

# undirected hyperedge
G.add_edge(src=['A', 'B', 'C'], edge_id='e_hyper', directed=False, relation='complex')

# directed hyperedge
G.add_edge(src=['A', 'B'], tgt=['D'], edge_id='e_dir_hyper', directed=True, relation='reaction')

# explicit stoichiometric incidence
G.add_edge(src={'A': 2.0, 'B': 1.0}, tgt={'D': 1.0}, edge_id='e_stoich', directed=True)

# edge-entity placeholder and edge->vertex relation
G.add_edge(edge_id='meta_e', as_entity=True, role='edge entity placeholder')
G.add_edge('meta_e', 'E', edge_id='e_meta_link', directed=True, edge_type='vertex_edge')

G.edges_view()


edge_id,kind,directed,global_weight,source,target,edge_type,head,tail,members,relation,role,effective_weight
str,str,bool,f64,str,str,str,list[str],list[str],list[str],str,str,f64
"""e_bin""","""binary""",true,1.0,"""A""","""B""","""binary""",null,null,null,"""activates""",null,1.0
"""e_hyper""","""hyper""",false,1.0,"""A|B|C""",null,null,null,null,"[""A"", ""B"", ""C""]","""complex""",null,1.0
"""e_dir_hyper""","""hyper""",true,1.0,"""A|B""","""D""",null,"[""A"", ""B""]","[""D""]",null,"""reaction""",null,1.0
"""e_stoich""","""hyper""",true,1.0,"""A|B""","""D""",null,"[""A"", ""B""]","[""D""]",null,null,null,1.0
"""e_meta_link""","""binary""",true,1.0,"""meta_e""","""E""","""binary""",null,null,null,null,null,1.0


## Managers and caches

The manager namespaces are now the primary public route for specialized behavior.


In [4]:
print('idx stats:', G.idx.stats())
print('cache info:', G.cache.info())
print('view edge count:', G.view(edges=['e_bin', 'e_hyper']).edge_count)


idx stats: {'n_entities': 6, 'n_vertices': 5, 'n_edge_entities': 1, 'n_edges': 5, 'max_row': 5, 'max_col': 4}
cache info: {'csr': {'cached': False}, 'csc': {'cached': False}, 'adjacency': {'cached': False}}
view edge count: 2
